In [11]:
import torch 
import torchvision
from torch.utils.data import DataLoader,Dataset,WeightedRandomSampler
import nibabel as nib
import pandas as   pd
from pathlib import Path
import wandb
import tqdm
import os
import torch.optim as optim
from collections import Counter
import numpy as np
import sys
import matplotlib.pyplot as plt
from sklearn.metrics import (
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
root = Path.cwd().parent
if str(root) not in sys.path:
    sys.path.append(str(root))

from MST.mst.data.datamodules.datamodule import DataModule
from MST.mst.data.datamodules.datamodule import DataModule
from MST.mst.data.datasets.dataset_3d_mrnet import MRNet_Dataset3D
from MST.mst.data.datasets.augmentations.augmentations_3d import SubjectToTensor,ImageOrSubjectToTensor, ImageToTensor,ZNormalization, RescaleIntensity,EnsureShapeMultiple
from MST.mst.models.dino import DinoV2ClassifierSlice

In [14]:
split=pd.read_csv(r'D:\Project\ACL\my_tool\MST_test_TN_data\preprocessed\splits\split.csv')
split.head()


,Unnamed: 0,ID,PatientKey,RawLabel,AclLabelName,abnormal,acl,meniscus,Plane,Folder,BaseSplit,Split,Fold,NrrdPath,SegPath,NiftiPath
0,0,0,2025_14_11_Thanh/Cai Kim Loc_25210566,Dut_ban_phan_1_3_giua,acl_injury,1,1,0,sagittal,train/,cross_validation_pool,train,1,D:\Project\ACL\my_tool\data\2025_14_11_Thanh\C...,D:\Project\ACL\my_tool\data\2025_14_11_Thanh\C...,D:\Project\ACL\my_tool\MST_test_TN_data\prepro...
1,1,1,2025_14_11_Thanh/Ha Van Chau_20250507,Dut_hoan_toan_1_3_giua,acl_injury,1,1,0,sagittal,train/,cross_validation_pool,train,0,D:\Project\ACL\my_tool\data\2025_14_11_Thanh\H...,D:\Project\ACL\my_tool\data\2025_14_11_Thanh\H...,D:\Project\ACL\my_tool\MST_test_TN_data\prepro...
2,2,2,2025_14_11_Thanh/Le Thi Thuy_25193052,Dut_ban_phan_1_3_duoi,acl_injury,1,1,0,sagittal,train/,cross_validation_pool,train,4,D:\Project\ACL\my_tool\data\2025_14_11_Thanh\L...,D:\Project\ACL\my_tool\data\2025_14_11_Thanh\L...,D:\Project\ACL\my_tool\MST_test_TN_data\prepro...
3,3,3,2025_14_11_Thanh/Le Van Chung_25938957,Dut_ban_phan_1_3_duoi,acl_injury,1,1,0,sagittal,train/,cross_validation_pool,train,1,D:\Project\ACL\my_tool\data\2025_14_11_Thanh\L...,D:\Project\ACL\my_tool\data\2025_14_11_Thanh\L...,D:\Project\ACL\my_tool\MST_test_TN_data\prepro...
4,4,4,2025_14_11_Thanh/Nguyen Hoang Gia Tuan_20250616,Dut_hoan_toan_1_3_giua,acl_injury,1,1,0,sagittal,train/,cross_validation_pool,train,0,D:\Project\ACL\my_tool\data\2025_14_11_Thanh\N...,D:\Project\ACL\my_tool\data\2025_14_11_Thanh\N...,D:\Project\ACL\my_tool\MST_test_TN_data\prepro...


In [ ]:
c1=split['Fold']==3
c2=split['acl']==0
print(split.loc[ c1 & c2])

In [28]:
class Datamodule_loader():
    
def split_dataset(path_root):
    split_csv_path=os.path.join(path_root,'preprocessed/splits/split.csv')
    df=pd.read_csv(split_csv_path)
    result=[]
    for i in range (5):
        fold_split_val=df.loc[df['Fold']==i].copy()
        fold_split_train=df.loc[df['Fold']!=i].copy()

        fold_split_val['Split']='val'
        fold_split_train['Split']='train'

        train_dataset=MRNet_Dataset3D(path_root=path_root,split='train',flip=True,random_rotate=True,noise=True)
        val_dataset=MRNet_Dataset3D(path_root=path_root,split='val',flip=False,random_rotate=False,noise=False)
        split_dict={'train_set':train_dataset,
                    'val_set':val_dataset}
        result.append(split_dict)
    return result

if __name__=="__main__":

    path_root='D:\Project\ACL\my_tool\MST_test_TN_data'

    fold_split_train_val=split_dataset(path_root=path_root)


In [33]:
print(fold_split_train_val[0]['train_set'][1]['source'].shape)

torch.Size([1, 32, 224, 224])


In [34]:
# for i in range(10):
#     print(test_dataset[i]['source'].shape)
# print(len(test_dataset))

In [20]:
# datamodule=DataModule(ds_train=train_dataset,
#                       ds_test=test_dataset,
#                       ds_val=val_dataset,batch_size=16,batch_size_test=16,batch_size_val=16)

# test_dataloader=datamodule.test_dataloader()

In [37]:
model = DinoV2ClassifierSlice(
    in_ch=1,
    out_ch=2,
    pretrained=True,
    model_size='s',
    use_registers=False,
    use_bottleneck=False,
    use_slice_pos_emb=True,
    slice_fusion='transformer',
    freeze=True,
    optimizer_kwargs={'lr': 1e-4, 'weight_decay': 1e-4},
)

Using cache found in C:\Users\hp/.cache\torch\hub\facebookresearch_dinov2_main


In [41]:
checkpoint_path=r'D:\Project\ACL\my_tool\MST_test_TN_data\checkpoints\best_revived-sweep-1_nu91b3wt.pt'
checkpoint=torch.load(checkpoint_path)
state_dict=checkpoint['model_state_dict']
model.load_state_dict(state_dict=state_dict)

<All keys matched successfully>

In [50]:
wandb.login()

wandb: Currently logged in as: baymaxnguyen306 (baymaxnguyen306-ho-chi-minh-city-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [42]:

class TrainerWB:
    def __init__(
        self,
        config,
        train_loader,
        val_loader,
        model:DinoV2ClassifierSlice,
        class_weights=None
    ):
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.epochs = config["epoch"]
        self.lr = config["lr"]

        # Regularization-related configs
        self.weight_decay = config.get("weight_decay", 1e-5)
        self.patience = config.get("patience", 15)
        self.label_smoothing = config.get("label_smoothing", 0.05)
        self.grad_clip = config.get("grad_clip", 1.0)
        
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.model = model.to(self.device)

        # Loss with label smoothing
        if class_weights is not None:
            self.criterion = nn.CrossEntropyLoss(
                weight=class_weights.to(self.device),
                label_smoothing=self.label_smoothing
            )
        else:
            self.criterion = nn.CrossEntropyLoss(
                label_smoothing=self.label_smoothing
            )

        # AdamW already includes decoupled weight decay
        self.optimizer = optim.AdamW(
            filter(lambda p: p.requires_grad, self.model.parameters()),
            lr=self.lr,
            weight_decay=self.weight_decay
        )

        # LR scheduler
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer=self.optimizer,
            mode='min',
            factor=0.5,
            patience=15
        )

        # Early stopping / best model
        self.best_macro_f1 = -1.0
        self.best_val_loss = float("inf")
        self.best_epoch = 0
        self.epochs_no_improve = 0
        self.best_state_dict = None
    
    def train_one_epoch(self):
        self.model.train()
        
        running_loss = 0.0
        correct = 0
        total = 0

        loop = tqdm(self.train_loader, desc="Training...", leave=False)

        for batch in loop:
            images = batch['source'].to(self.device, non_blocking=True)
            labels = batch['target'].to(self.device, non_blocking=True)

            self.optimizer.zero_grad()

            outputs = self.model(images)
            loss = self.criterion(outputs, labels)

            loss.backward()

            # Gradient clipping
            if self.grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=self.grad_clip)

            self.optimizer.step()

            batch_size = images.size(0)
            running_loss += loss.item() * batch_size

            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += batch_size

            loop.set_postfix(
                loss=running_loss / total,
                acc=correct / total,
                lr=self.optimizer.param_groups[0]["lr"]
            )

        train_loss = running_loss / total
        train_acc = correct / total
        return train_loss, train_acc

    def validate_one_epoch(self):
        self.model.eval()

        running_loss = 0.0
        correct = 0
        total = 0

        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch in self.val_loader:
                images = batch['source'].to(self.device, non_blocking=True)
                labels = batch['target'].to(self.device, non_blocking=True)

                outputs = self.model(images)
                loss = self.criterion(outputs, labels)

                batch_size = images.size(0)
                running_loss += loss.item() * batch_size

                preds = outputs.argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += batch_size

                all_preds.append(preds.cpu())
                all_labels.append(labels.cpu())

            all_preds = torch.cat(all_preds).numpy()
            all_labels = torch.cat(all_labels).numpy()

        val_loss = running_loss / total
        val_acc = correct / total

        macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
        weighted_f1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
        cm = confusion_matrix(all_labels, all_preds)

        return val_loss, val_acc, macro_f1, weighted_f1, cm

    def save_best_model(self, save_path="best_model.pth"):
        if self.best_state_dict is not None:
            torch.save(self.best_state_dict, save_path)

    def fit(self, print_every=1, cm_every=5):
        print(f"Using device: {self.device}")

        for epoch in range(1, self.epochs + 1):
            train_loss, train_acc = self.train_one_epoch()
            val_loss, val_acc, macro_f1, weighted_f1, cm = self.validate_one_epoch()

            self.scheduler.step(val_loss)
            current_lr = self.optimizer.param_groups[0]["lr"]

            if epoch % print_every == 0:
                print(f"\nEpoch [{epoch}/{self.epochs}]")
                print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
                print(f"Val Loss:   {val_loss:.4f}, Val Acc:   {val_acc:.4f}")
                print(f"Macro F1:   {macro_f1:.4f}")
                print(f"Weighted F1:{weighted_f1:.4f}")
                print(f"LR: {current_lr:.6f}")
                print(f"Best Macro F1 so far: {self.best_macro_f1:.4f} at epoch {self.best_epoch}")

            if epoch % cm_every == 0:
                print("\nConfusion Matrix:")
                print(cm)

            wandb.log({
                "epoch": epoch,
                "lr": current_lr,
                "train_loss": train_loss,
                "train_accuracy": train_acc,
                "val_loss": val_loss,
                "val_accuracy": val_acc,
                "val_macro_f1": macro_f1,
                "val_weighted_f1": weighted_f1,
                "best_macro_f1": self.best_macro_f1,
                'cm':cm
            })

            if self.epochs_no_improve >= self.patience:
                print(f"\nEarly stopping triggered at epoch {epoch}.")
                break

        print(f"\nTraining complete.")
        print(f"Best Macro F1: {self.best_macro_f1:.4f} at epoch {self.best_epoch}")

In [47]:
from collections import Counter
import numpy as np
import torch
from torch.utils.data import WeightedRandomSampler


class Sampler:
    def __init__(self, dataset):
        self.dataset = dataset
        self.labels = self._extract_labels_once()
        self.class_counts = self._get_class_counts()

    def _extract_labels_once(self):
        labels = []

        for i in range(len(self.dataset)):
            sample = self.dataset[i]
            label = sample["target"]

            if isinstance(label, torch.Tensor):
                label = label.item()

            labels.append(int(label))

        return labels

    def _get_class_counts(self):
        class_counts = Counter(self.labels)
        return [class_counts[k] for k in sorted(class_counts.keys())]

    def init_sampler(self):
        weights = 1.0 / np.array(self.class_counts, dtype=np.float32)
        sample_weights = np.array(
            [weights[label] for label in self.labels],
            dtype=np.float32
        )

        sampler = WeightedRandomSampler(
            weights=torch.from_numpy(sample_weights),
            num_samples=len(sample_weights),
            replacement=True
        )
        return sampler

# if __name__=="__main__":
#     train_dataset=fold_split_train_val[1]['train_set']
#     s=Sampler(train_dataset)
#     sampler=s.init_sampler()

In [48]:
def main(train_dataset,val_dataset):
        with wandb.init(project="DinoV2_fold_{i}", reinit=True):
            s=Sampler(train_dataset)
            sampler=s.init_sampler()
            config=wandb.config
            train_loader = DataLoader(train_dataset,batch_size=config.batch_size,sampler=sampler)
            val_loader = DataLoader(val_dataset,batch_size=config.batch_size,shuffle=False)
            trainer = TrainerWB(
                config=wandb.config,
                train_loader=train_loader,
                val_loader=val_loader,
                model=model
            )
            trainer.fit(print_every=5,cm_every=5)

In [49]:
sweep_config={
    'method':'random',
    'metric':{
        'name':'val_weighted_f1',
        'goal':'maximize'
    },
    'parameters':{
        'lr':{
            'values':[1e-4,1e-5]
        },
        'epoch':{
            'values':[30,40,50]
        },
        'batch_size':{
            'values':[4,8,16]
        },
        'weight_decays':{
            'values':[1e-4,1e-5]
        }
    }
}

In [ ]:
for i in range(5):
    train_dataset=fold_split_train_val[i]['train_set']
    val_dataset=fold_split_train_val[i]['val_set']
    sweep_id= wandb.sweep(sweep_config, project="DinoV2_fold_{i}")
    wandb.agent(sweep_id,  function=main(train_dataset=train_dataset,val_dataset=val_dataset), count=2)

In [40]:
class Trainer:
    def __init__(self,model:DinoV2ClassifierSlice, 
                 config,
                checkpoint_path:Path,
                train_loader:DataLoader,
                val_loader:DataLoader
                device="cuda"
                ):
        self.epoch=config['epoch']
        self.lr=config['lr']
        self.batch_size=config['batch_size']
        self.criterion=torch.nn.CrossEntropyLoss()
        self.model=model
        self.checkpoint_path=checkpoint_path
        self.train_loader=train_loader
        self.val_loader=val_loader
    
    def loading_state_dict(self,model,statedict):
        return (model.load_state_dict(statedict))
    
    def train_one_epoch(self):
        pass

    def validate(self):
        pass

    
        

SyntaxError: invalid syntax. Perhaps you forgot a comma? (3333680198.py, line 6)

In [27]:
from pathlib import Path
import torch
import tqdm
import numpy as np

class Evaluator:
    def __init__(
        self,
        model,
        test_loader,
        checkpoint_folder: Path,
        device="cuda",
        criterion=torch.nn.CrossEntropyLoss(),
    ):
        self.test_loader = test_loader
        self.checkpoint_folder = checkpoint_folder
        self.model = model
        self.device = device
        self.criterion = criterion
        self.batch_size = test_loader.batch_size

    def loading_model_state_dict(self, model, statedict):
        model.load_state_dict(statedict)
        return model

    def evaluate(self):
        inference_result = []

        for pth_file in self.checkpoint_folder.glob("*.pt"):
            print(f"Loading checkpoint: {pth_file.name}")

            checkpoint = torch.load(pth_file, map_location=self.device)

            # if checkpoint is a full training checkpoint
            if "model_state_dict" in checkpoint:
                state_dict = checkpoint["model_state_dict"]
            else:
                state_dict = checkpoint  # in case some files are pure state_dict files

            self.model = self.loading_model_state_dict(self.model, state_dict)
            self.model = self.model.to(self.device)
            self.model.eval()
            # initialize metric accumulators
            running_loss = 0.0
            correct = 0
            total = 0
            all_preds = []
            all_labels = []

            loop = tqdm.tqdm(self.test_loader, desc=f"Inferencing {pth_file.name}")
            print(f"Using device: {self.device}")

            with torch.no_grad():
                for batch in loop:
                    images = batch["source"].to(self.device)
                    labels = batch["target"].to(self.device)

                    outputs = self.model(images)
                    loss = self.criterion(outputs, labels)

                    batch_size = images.size(0)
                    running_loss += loss.item() * batch_size

                    preds = outputs.argmax(dim=1)
                    total += batch_size
                    correct += (preds == labels).sum().item()

                    all_preds.append(preds.cpu())
                    all_labels.append(labels.cpu())

            all_preds = torch.cat(all_preds).numpy()
            all_labels = torch.cat(all_labels).numpy()

            avg_loss = running_loss / total
            acc = correct / total
            macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
            weighted_f1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
            balanced_acc = balanced_accuracy_score(all_labels, all_preds)
            cm = confusion_matrix(all_labels, all_preds)

            checkpoint_inference_result = {
                "checkpoint": pth_file.name,
                "loss": avg_loss,
                "accuracy": acc,
                "macro_f1": macro_f1,
                "weighted_f1": weighted_f1,
                "balanced_acc": balanced_acc,
                "cm": cm,
            }

            inference_result.append(checkpoint_inference_result)

        return inference_result

    def visualize_result(self, result, class_names=None, save_dir=None, figsize=(8, 6)):
        """
        Visualize evaluation results for every checkpoint.

        Args:
            result (list[dict]): output from self.evaluate()
            class_names (list[str] | None): names of classes for confusion matrix axes
            save_dir (str | Path | None): folder to save figures; if None, just show them
            figsize (tuple): figure size for confusion matrix
        """
        if len(result) == 0:
            print("No result to visualize.")
            return

        if save_dir is not None:
            save_dir = Path(save_dir)
            save_dir.mkdir(parents=True, exist_ok=True)

        # -------- Summary bar chart across all checkpoints --------
        checkpoint_names = [r["checkpoint"] for r in result]
        macro_f1s = [r["macro_f1"] for r in result]
        weighted_f1s = [r["weighted_f1"] for r in result]
        balanced_accs = [r["balanced_acc"] for r in result]
        accuracies = [r.get("accuracy", None) for r in result]

        x = np.arange(len(checkpoint_names))
        width = 0.2

        plt.figure(figsize=(max(10, len(checkpoint_names) * 1.5), 6))
        plt.bar(x - 1.5 * width, macro_f1s, width, label="Macro F1")
        plt.bar(x - 0.5 * width, weighted_f1s, width, label="Weighted F1")
        plt.bar(x + 0.5 * width, balanced_accs, width, label="Balanced Acc")

        if all(a is not None for a in accuracies):
            plt.bar(x + 1.5 * width, accuracies, width, label="Accuracy")

        plt.xticks(x, checkpoint_names, rotation=45, ha="right")
        plt.ylim(0, 1.0)
        plt.ylabel("Score")
        plt.title("Checkpoint Performance Comparison")
        plt.legend()
        plt.tight_layout()

        if save_dir is not None:
            plt.savefig(save_dir / "checkpoint_metric_comparison.png", dpi=300, bbox_inches="tight")
            plt.close()
        else:
            plt.show()

        # -------- Confusion matrix for each checkpoint --------
        for r in result:
            cm = r["cm"]
            checkpoint_name = r["checkpoint"]

            fig, ax = plt.subplots(figsize=figsize)
            disp = ConfusionMatrixDisplay(
                confusion_matrix=cm,
                display_labels=class_names if class_names is not None else None,
            )
            disp.plot(ax=ax, cmap="Blues", colorbar=False)

            ax.set_title(
                f"{checkpoint_name}\n"
                f"Macro F1={r['macro_f1']:.4f} | "
                f"Weighted F1={r['weighted_f1']:.4f} | "
                f"Balanced Acc={r['balanced_acc']:.4f}"
            )
            plt.tight_layout()

            if save_dir is not None:
                safe_name = checkpoint_name.replace(".pth", "").replace("/", "_").replace("\\", "_")
                plt.savefig(save_dir / f"{safe_name}_confusion_matrix.png", dpi=300, bbox_inches="tight")
                plt.close()
            else:
                plt.show()

In [28]:
evaluator = Evaluator(
    model=model,
    test_loader=test_dataloader,
    checkpoint_folder=Path("./checkpoints"),
    device="cuda"
)

results = evaluator.evaluate()

evaluator.visualize_result(
    result=results,
    class_names=["class_0", "class_1"],   # change this to your real class names
    save_dir="./eval_figures"
)

Loading checkpoint: best_generous-sweep-5_yg4ngc5e.pt


Inferencing best_generous-sweep-5_yg4ngc5e.pt:   0%|          | 0/11 [00:00<?, ?it/s]d:\Project\ACL\.venv\Lib\site-packages\torchio\transforms\transform.py:168: RuntimeWarning: Output shape (225, 225, 32) != target shape (np.int64(224), np.int64(224), np.int64(32)). Fixing with CropOrPad
  transformed = self.apply_transform(subject)


Using device: cuda


d:\Project\ACL\.venv\Lib\site-packages\torchio\transforms\transform.py:168: RuntimeWarning: Output shape (225, 224, 32) != target shape (np.int64(224), np.int64(224), np.int64(32)). Fixing with CropOrPad
  transformed = self.apply_transform(subject)
d:\Project\ACL\.venv\Lib\site-packages\torchio\transforms\transform.py:168: RuntimeWarning: Output shape (224, 225, 32) != target shape (np.int64(224), np.int64(224), np.int64(32)). Fixing with CropOrPad
  transformed = self.apply_transform(subject)
Inferencing best_generous-sweep-5_yg4ngc5e.pt: 100%|██████████| 11/11 [01:59<00:00, 10.86s/it]


Loading checkpoint: best_hopeful-sweep-2_9sislb46.pt


...

d:\Project\ACL\.venv\Lib\site-packages\torchio\transforms\transform.py:168: RuntimeWarning: Output shape (225, 225, 32) != target shape (np.int64(224), np.int64(224), np.int64(32)). Fixing with CropOrPad
  transformed = self.apply_transform(subject)
d:\Project\ACL\.venv\Lib\site-packages\torchio\transforms\transform.py:168: RuntimeWarning: Output shape (225, 224, 32) != target shape (np.int64(224), np.int64(224), np.int64(32)). Fixing with CropOrPad
  transformed = self.apply_transform(subject)
d:\Project\ACL\.venv\Lib\site-packages\torchio\transforms\transform.py:168: RuntimeWarning: Output shape (224, 225, 32) != target shape (np.int64(224), np.int64(224), np.int64(32)). Fixing with CropOrPad
  transformed = self.apply_transform(subject)
Inferencing best_serene-sweep-4_bi4er13r.pt: 100%|██████████| 11/11 [01:53<00:00, 10.35s/it]


Loading checkpoint: best_vocal-sweep-2_211f7gd9.pt


Inferencing best_vocal-sweep-2_211f7gd9.pt:   0%|          | 0/11 [00:00<?, ?it/s]

Using device: cuda


d:\Project\ACL\.venv\Lib\site-packages\torchio\transforms\transform.py:168: RuntimeWarning: Output shape (225, 225, 32) != target shape (np.int64(224), np.int64(224), np.int64(32)). Fixing with CropOrPad
  transformed = self.apply_transform(subject)
d:\Project\ACL\.venv\Lib\site-packages\torchio\transforms\transform.py:168: RuntimeWarning: Output shape (225, 224, 32) != target shape (np.int64(224), np.int64(224), np.int64(32)). Fixing with CropOrPad
  transformed = self.apply_transform(subject)
d:\Project\ACL\.venv\Lib\site-packages\torchio\transforms\transform.py:168: RuntimeWarning: Output shape (224, 225, 32) != target shape (np.int64(224), np.int64(224), np.int64(32)). Fixing with CropOrPad
  transformed = self.apply_transform(subject)
Inferencing best_vocal-sweep-2_211f7gd9.pt: 100%|██████████| 11/11 [01:53<00:00, 10.33s/it]
